# Freegle Mod Helper - QLoRA Fine-tuning (Experimental)

Fine-tunes **Qwen2.5-3B-Instruct** on Freegle moderation data using **QLoRA** (4-bit quantisation + LoRA adapters). This is an **experimental investigation** into whether AI can usefully assist human moderators — not a commitment to any particular approach.

**Requirements**: Google Colab free tier with T4 GPU (15 GB VRAM).

## Estimated Time Budget

| Step | Estimated Time |
|------|---------------|
| 1. Setup & Install | 3-5 min |
| 2. Mount Google Drive | < 1 min |
| 3. Load & Prepare Data | 1-2 min |
| 4. Load Model (4-bit) | 2-3 min |
| 5. Add LoRA Adapters | < 1 min |
| 6. Training (1 epoch, 39k examples packed) | 2-3 hours |
| 7. Export to GGUF | 10-15 min |
| 8. Create Ollama Modelfile | < 1 min |
| 9. Quick Validation | 1-2 min |
| **Total** | **~3-4 hours** |

Checkpoints save to Google Drive every 100 steps. If disconnected, re-run and it resumes automatically.

## 1. Setup

Install unsloth and dependencies. Pinned versions known to work on Colab free tier with T4.

**Estimated time: 3-5 minutes**

In [ ]:
# Install unsloth - use the recommended Colab method.
# IMPORTANT: Do NOT reinstall torch — Colab's pre-installed torch+CUDA combo must stay intact.
!pip install unsloth
# Get latest unsloth without pulling deps (avoids breaking torch/CUDA)
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

# Verify critical imports work
import torch
import transformers
import peft
import trl
import bitsandbytes
print(f"torch:          {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"transformers:   {transformers.__version__}")
print(f"peft:           {peft.__version__}")
print(f"trl:            {trl.__version__}")
print(f"bitsandbytes:   {bitsandbytes.__version__}")
print("All imports OK.")

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "No GPU detected — enable T4 via Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2. Mount Google Drive

Training data should be at: `My Drive/freegle-modbot/data/combined_train.jsonl`

Model output will be saved to: `My Drive/freegle-modbot/models/`

**Estimated time: < 1 minute**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Paths — adjust these if your Drive layout differs
DRIVE_BASE = "/content/drive/MyDrive/freegle-modbot"
DATA_DIR = f"{DRIVE_BASE}/data"
MODEL_DIR = f"{DRIVE_BASE}/models"
TRAIN_FILE = f"{DATA_DIR}/combined_train.jsonl"

# Create output directory
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify training data exists
assert os.path.isfile(TRAIN_FILE), (
    f"Training data not found at {TRAIN_FILE}\n"
    f"Upload combined_train.jsonl to Google Drive at: freegle-modbot/data/"
)

file_size_mb = os.path.getsize(TRAIN_FILE) / 1e6
print(f"Training data: {TRAIN_FILE} ({file_size_mb:.1f} MB)")
print(f"Model output:  {MODEL_DIR}")

## 3. Load & Prepare Data

Load the JSONL, strip the `meta` field, and format each example using the Qwen ChatML template.

Three task types in the data:
- **moderation** (18,056 examples): Approve/reject post decisions
- **text_cleanup** (12,000 examples): Remove personal info, selling language, etc.
- **subject_correction** (9,205 examples): Fix spelling/formatting in post subjects

**Estimated time: 1-2 minutes**

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a Freegle community moderator assistant. "
    "You review posts for rule violations, correct spelling, "
    "and flag inappropriate content."
)

def format_chatml(example):
    """Format a single example as a Qwen ChatML conversation."""
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": text}

# Load JSONL and strip meta field
raw_data = []
task_counts = {}
with open(TRAIN_FILE, "r") as f:
    for line in f:
        row = json.loads(line)
        task = row.get("meta", {}).get("task", "unknown")
        task_counts[task] = task_counts.get(task, 0) + 1
        raw_data.append({
            "instruction": row["instruction"],
            "output": row["output"],
        })

print(f"Loaded {len(raw_data):,} examples")
for task, count in sorted(task_counts.items(), key=lambda x: -x[1]):
    print(f"  {task}: {count:,}")

# Convert to HF Dataset and apply chat template
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chatml, remove_columns=["instruction", "output"])

print(f"\nDataset: {len(dataset):,} rows, 1 column ('text')")
print(f"\n--- Sample (first 500 chars) ---")
print(dataset[0]["text"][:500])

## 4. Load Model

Load Qwen2.5-3B-Instruct with 4-bit quantisation via unsloth. This uses ~4 GB VRAM.

**Estimated time: 2-3 minutes**

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

# Load the base model with 4-bit quantisation
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (float16 on T4)
    load_in_4bit=True,
)

print(f"Base model loaded: Qwen2.5-3B-Instruct (4-bit)")

# VRAM check
gpu_mem_used = torch.cuda.memory_allocated() / 1e9
gpu_mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM after load: {gpu_mem_used:.1f} / {gpu_mem_total:.1f} GB")

## 5. Add LoRA Adapters

Add low-rank adapters to the frozen base model. Only the adapter weights are trained.

- **r=16**: Rank of the low-rank matrices
- **lora_alpha=32**: Scaling factor (alpha/r = 2.0)
- **lora_dropout=0.05**: Regularisation during training
- **Target modules**: All attention projections (q, k, v, o) + MLP layers (gate, up, down)

**Estimated time: < 1 minute**

In [ ]:
# Add LoRA adapters to the base model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

# Print trainable parameter summary
model.print_trainable_parameters()

# Verify LoRA config
print("\nLoRA Configuration:")
print(f"  Rank (r):       {model.peft_config['default'].r}")
print(f"  Alpha:          {model.peft_config['default'].lora_alpha}")
print(f"  Dropout:        {model.peft_config['default'].lora_dropout}")
print(f"  Target modules: {model.peft_config['default'].target_modules}")
print(f"  Bias:           {model.peft_config['default'].bias}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

# VRAM check
gpu_mem_used = torch.cuda.memory_allocated() / 1e9
gpu_mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nVRAM: {gpu_mem_used:.1f} / {gpu_mem_total:.1f} GB")

## 6. Train

Fine-tune using SFTTrainer with the formatted ChatML dataset.

Effective batch size = `per_device_train_batch_size` (4) x `gradient_accumulation_steps` (4) = **16**

With packing, ~39k examples compress to ~3,700 packed sequences → ~232 steps per epoch.

**1 epoch is sufficient** — loss reaches ~0.73 which is good for this task. More epochs give diminishing returns and risk Colab session timeout.

Checkpoints save to **Google Drive** so they survive disconnects.

**Estimated time: 2-3 hours**

In [ ]:
import time

# Save checkpoints to Google Drive (survives disconnects)
CHECKPOINT_DIR = f"{MODEL_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Check for existing checkpoint to resume from
import glob
existing_checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"))
resume_from = existing_checkpoints[-1] if existing_checkpoints else None
if resume_from:
    print(f"Found existing checkpoint: {resume_from}")
    print("Will resume training from this checkpoint.\n")
else:
    print("No existing checkpoints found. Starting fresh.\n")

# Use SFTConfig (newer trl API) if available, fall back to TrainingArguments
try:
    from trl import SFTTrainer, SFTConfig
    training_config = SFTConfig(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=10,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        disable_tqdm=False,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=True,
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=training_config,
    )
    print("Using SFTConfig (newer trl API)")
except (ImportError, TypeError):
    from trl import SFTTrainer
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=10,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        disable_tqdm=False,
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
        args=training_args,
    )
    print("Using TrainingArguments (older trl API)")

# Print training plan
print(f"\nTraining plan:")
print(f"  Examples:       {len(dataset):,}")
print(f"  Batch size:     4 x 4 = 16 effective")
print(f"  Epochs:         1")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Packing:        enabled")
print(f"  Precision:      fp16")
print(f"  Checkpoints:    every 100 steps → Google Drive")
print(f"  Resume from:    {resume_from or 'scratch'}")
print()

# Train with timing
start = time.time()
train_result = trainer.train(resume_from_checkpoint=resume_from)
elapsed = time.time() - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
print(f"\nTraining complete in {hours}h {minutes}m")
print(f"Final loss: {train_result.training_loss:.4f}")

# Save final model to Drive
print(f"\nSaving final model to Drive...")
trainer.save_model(f"{CHECKPOINT_DIR}/final")
print("Done.")

In [ ]:
# Training loss curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [x["step"] for x in log_history if "loss" in x]
losses = [x["loss"] for x in log_history if "loss" in x]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, linewidth=0.8)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss - Freegle Mod Helper QLoRA")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Start loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")

## 7. Export to GGUF

Merge LoRA adapters into the base model, quantise to Q4_K_M, and save the GGUF file to Google Drive.

Q4_K_M is a good balance: ~2 GB file size, minimal quality loss vs fp16.

**Estimated time: 10-15 minutes**

In [ ]:
GGUF_DIR = f"{MODEL_DIR}/freegle-mod-helper-q4_k_m"

print("Exporting to GGUF (this compiles llama.cpp — may take 10-15 minutes)...")
print("Progress will appear below.\n")

# Save merged model as GGUF with Q4_K_M quantisation
try:
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method="q4_k_m",
    )
    # List output files
    print(f"\nSaved to: {GGUF_DIR}")
    for f in os.listdir(GGUF_DIR):
        size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")
except Exception as e:
    print(f"\nGGUF export failed: {e}")
    print("\nFalling back to saving as fp16 GGUF (no quantisation)...")
    try:
        model.save_pretrained_gguf(
            GGUF_DIR,
            tokenizer,
            quantization_method="f16",
        )
        print("Saved as fp16 (larger file, but works)")
        for f in os.listdir(GGUF_DIR):
            size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e6
            print(f"  {f} ({size_mb:.1f} MB)")
    except Exception as e2:
        print(f"fp16 export also failed: {e2}")
        print("LoRA adapters were saved separately (cell below). You can merge+quantise locally.")

In [ ]:
# Also save the LoRA adapters separately (smaller, can re-merge later)
LORA_DIR = f"{MODEL_DIR}/freegle-mod-helper-lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA adapters saved to: {LORA_DIR}")

## 8. Create Ollama Modelfile

Generate a Modelfile for importing the GGUF into Ollama on your local machine.

**Estimated time: < 1 minute**

In [ ]:
# Find the GGUF file name
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
assert len(gguf_files) > 0, f"No .gguf files found in {GGUF_DIR}"
gguf_filename = gguf_files[0]

MODELFILE_LINES = [
    "# Freegle Mod Helper - Ollama Modelfile",
    "# Qwen2.5-3B-Instruct fine-tuned with QLoRA on Freegle moderation data",
    "# Experimental - assists moderators, does not replace them",
    "#",
    "# Usage:",
    "#   1. Copy the .gguf file and this Modelfile to the same directory",
    "#   2. Run: ollama create freegle-mod-helper -f Modelfile",
    "#   3. Test: ollama run freegle-mod-helper",
    "",
    f"FROM ./{gguf_filename}",
    "",
    'TEMPLATE """<|im_start|>system',
    "{{ .System }}<|im_end|>",
    "<|im_start|>user",
    "{{ .Prompt }}<|im_end|>",
    "<|im_start|>assistant",
    '{{ .Response }}<|im_end|>"""',
    "",
    'SYSTEM """You are a Freegle community moderator assistant. You help moderators by reviewing posts for rule violations, correcting spelling, and flagging inappropriate content. Your suggestions are advisory — the moderator makes the final decision."""',
    "",
    "PARAMETER temperature 0.1",
    "PARAMETER num_ctx 2048",
    'PARAMETER stop "<|im_end|>"',
    'PARAMETER stop "<|im_start|>"',
    "",
]

MODELFILE_CONTENT = "\n".join(MODELFILE_LINES)

modelfile_path = os.path.join(GGUF_DIR, "Modelfile")
with open(modelfile_path, "w") as f:
    f.write(MODELFILE_CONTENT)

print(f"Modelfile saved to: {modelfile_path}")
print()
print("--- Modelfile contents ---")
print(MODELFILE_CONTENT)

## 9. Quick Validation

Run 5 sample inferences to verify the fine-tuned model produces reasonable outputs.

**Estimated time: 1-2 minutes**

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

test_prompts = [
    # 1. Moderation - should approve
    (
        "You are a Freegle community moderator. Review this post and decide whether to approve or reject it.\n\n"
        "Group: BristolFreegle\n"
        "Post type: Offer\n"
        "Subject: OFFER: Wooden bookshelf (Clifton BS8)\n"
        "Body: Solid pine bookshelf, 5 shelves. Good condition, some scratches. Collection only."
    ),
    # 2. Moderation - should reject (selling)
    (
        "You are a Freegle community moderator. Review this post and decide whether to approve or reject it.\n\n"
        "Group: LeedsFreegle\n"
        "Post type: Offer\n"
        "Subject: OFFER: iPhone 12 (Leeds LS1)\n"
        "Body: Selling my iPhone 12. Asking 150 pounds. Text me on 07712345678."
    ),
    # 3. Subject correction
    (
        "Fix any spelling, formatting, or content errors in this Freegle post subject. "
        "Only fix clear errors, don't change meaning. If the subject is correct, return it unchanged.\n\n"
        "Subject: OFFRE: Childrens bycycle (manchster M14)"
    ),
    # 4. Text cleanup - personal info
    (
        "Clean up this Freegle post text. Remove: personal info (phone numbers, full postcodes, addresses), "
        "selling/pricing language, borrowing requests, excessive sob stories. Keep the item description intact.\n\n"
        "Text: Large garden table and 4 chairs. Call me on 07891234567 or email john.smith@gmail.com. "
        "I live at 42 Oak Street, Bristol BS8 1AB. Pick up anytime."
    ),
    # 5. Moderation - edge case (vague post)
    (
        "You are a Freegle community moderator. Review this post and decide whether to approve or reject it.\n\n"
        "Group: OxfordFreegle\n"
        "Post type: Wanted\n"
        "Subject: WANTED: Stuff (Oxford OX1)\n"
        "Body: Looking for anything really"
    ),
]

print("=" * 80)
print("VALIDATION: 5 sample inferences")
print("=" * 80)

for i, prompt in enumerate(test_prompts, 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens (skip the prompt)
    response = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    )

    print(f"\n--- Test {i} ---")
    # Show just the first line of the prompt for brevity
    prompt_preview = prompt.split("\n")[0][:80]
    print(f"Prompt: {prompt_preview}...")
    print(f"Response:\n{response.strip()}")
    print()

print("=" * 80)
print("Validation complete. Review outputs above for quality.")
print(f"\nGGUF model saved at: {GGUF_DIR}")
print(f"To use with Ollama: copy the .gguf + Modelfile, then run:")
print(f"  ollama create freegle-mod-helper -f Modelfile")